# Tutorial: Preparing submission file and run evaluation for Task B

In this notebook, a step-by-step tutorial is provided for preparing the submission file for the shared task Task B. To achieve this, the data for Task B, hosted on [Zenodo](https://doi.org/10.5281/zenodo.14002665), will be downloaded; a file with the appropriate [submission format](https://talentclef.github.io/talentclef/docs/talentclef-2025/evaluation/) will be prepared, and it will be evaluated using the [task's evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script). Additionally, the provided format is also compatible with the benchmark where the test set data will be uploaded on Codabench.



-----------------------------
TalentCLEF is an initiative to advance Natural Language Processing (NLP) in Human Capital Management (HCM). It aims to create a public benchmark for model evaluation and promote collaboration to develop fair, multilingual, and flexible systems that improve Human Resources (HR) practices across different industries.

This shared-task's inaugural edition is part of the [Conference and Labs of the Evaluation Forum (CLEF)](https://clef2025.clef-initiative.eu/index.php?page=Pages/labs.html), scheduled to be held in Madrid in 2025. If you are interested in registering, you can find registration form [here](https://clef2025-labs-registration.dei.unipd.it/).

<img src="https://github.com/TalentCLEF/talentclef/blob/main/logo_talentclef.png?raw=true" alt="TalentCLEF logo" width="200"/>
<img src="https://talentclef.github.io/talentclef/docs/talentclef-2025/workshop/logo_clef_madrid.png" alt="TalentCLEF logo" width="150"/>


## Imports

In [20]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer, util
import subprocess
from codecarbon import EmissionsTracker

In [2]:
import os
import sys

## Download Task B files

First, let's download the Task A and Task B zip files directly from Zenodo.



In [ ]:
# Download
!wget https://zenodo.org/records/15038364/files/TaskB.zip
!unzip TaskB.zip -d taskB

## Generate releveant files using a simple model

**DE MOMENTO SUBIR LOS DATOS A MANOS HASTA EL RELEASE.**

In [3]:
taskB_path="/Users/zia-mac/Documents/projects/data/ir_data/talent_clef/TaskB"
taskB_validation_path=os.path.join(taskB_path, "validation")

In [28]:
taskB_models_path=os.path.join(taskB_path, "models")
taskB_results_path=os.path.join(taskB_path, "results")
taskB_evaluations_path=os.path.join(taskB_path, "evaluations")

model_name="all-MiniLM-L6-v2"
sentence_transformer_model=os.path.join(taskB_models_path, model_name)

Load queries and corpus elements in English from the Validation folder:

In [4]:
queries = os.path.join(taskB_validation_path, "queries")
corpus_elements = os.path.join(taskB_validation_path, "corpus_elements")

In [5]:
queries = pd.read_csv(queries,sep="\t")
corpus_elements = pd.read_csv(corpus_elements, sep="\t")


Transform `skill_aliases` column to a list of strings:

In [6]:
import ast
corpus_elements["skill_aliases"] = corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))

Generate a mapping dictionary between IDs and texts from query and corpus element strings.

In [7]:
queries_ids = queries.q_id.to_list()
queries_texts = queries.jobtitle.to_list()
map_queries = dict(zip(queries_ids,queries_texts))

Before creating a mapping dictionary of texts to corpus_ids, explode the `skill_aliases` column:

In [8]:
list_aliases_df = corpus_elements.explode("skill_aliases")

In [10]:
corpus_ids = list_aliases_df.c_id.to_list()
corpus_texts = list_aliases_df.skill_aliases.to_list()
map_corpus = dict(zip(corpus_texts,corpus_ids))
map_corpus

{'pricing plans': 'dev_cb_sk_1',
 'price strategies': 'dev_cb_sk_1',
 'pricing tactics': 'dev_cb_sk_1',
 'pricing strategies': 'dev_cb_sk_1',
 'pricing strategy': 'dev_cb_sk_1',
 'putting out fires': 'dev_cb_sk_2',
 'put out fires': 'dev_cb_sk_2',
 'extinguish fires': 'dev_cb_sk_2',
 'fire extinguishing': 'dev_cb_sk_2',
 'fires extinguishing': 'dev_cb_sk_2',
 'extinguishing fires': 'dev_cb_sk_2',
 'fire putting out': 'dev_cb_sk_2',
 'fires putting out': 'dev_cb_sk_2',
 'online assessment': 'dev_cb_sk_3',
 'analysis of web strategy': 'dev_cb_sk_3',
 'web presence assessment': 'dev_cb_sk_3',
 'web strategy assessment': 'dev_cb_sk_3',
 'keeping up with trends': 'dev_cb_sk_4',
 'keep pace with trends': 'dev_cb_sk_4',
 'follow trends': 'dev_cb_sk_4',
 'keep up with a trend': 'dev_cb_sk_4',
 'keeping up with a trend': 'dev_cb_sk_4',
 'monitor trends': 'dev_cb_sk_4',
 'keep up with trends': 'dev_cb_sk_4',
 'prepare tax returns form': 'dev_cb_sk_5',
 'complete tax returns forms': 'dev_cb_sk_5'

Load simple embedding model:

In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2", token=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
c 

In [13]:
tracker = EmissionsTracker()
tracker.start_task("all-MiniLM-L6-v2")

Encode queries and corpus elements:

In [14]:
query_embeddings = model.encode(queries_texts, convert_to_tensor=True)
corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True)

Compute similarities

In [15]:
similarities = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()
emissions = tracker.stop_task("all-MiniLM-L6-v2")

## Prepare submission file

The submissions must follow the TREC Run File format, including headers in the output file. This means that the fle have 6 space-spearated columns per line, with following information:

- q_id: Query ID.
- Q0: A constant identifier, usually "Q0".
- doc_id: ID of the retrieved document.
- rank: Position of the document in the ranking.
- score: Relevance score assigned by the model.
- tag: Experiment name

In [18]:
import numpy as np
results = []
results_name = []

for q_idx, q_id in enumerate(queries_ids):
    sorted_indices = np.argsort(-similarities[q_idx])
    used_doc_ids = set()
    rank_counter = 0
    for c_idx in sorted_indices:  # Consider the full list.
        doc_id = corpus_ids[c_idx]
        # If doc_id was already processed, go to the next one.
        if doc_id in used_doc_ids:
            continue
        used_doc_ids.add(doc_id)
        rank_counter += 1

        query_name = map_queries[q_id]
        doc_name = corpus_texts[c_idx]
        score = similarities[q_idx, c_idx]

        results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} baseline_model")
        results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} baseline_model")

The list has this structure

Let's save the list as a file:

In [21]:
result_baseline_taskB = os.path.join(taskB_results_path, "evaluation_baseline_taskB.trec")
with open(result_baseline_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results))

emissions_path = os.path.join(taskB_evaluations_path, "emissions.json")
json.dump(dict(emissions.values), open(emissions_path, "w"), ensure_ascii=False, indent=4)

## Evaluation

For the evaluation, we will use the official [TalentCLEF evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script), which uses the Ranx library under the hood.

First, clone the repo and install the requirements file:

In [22]:
!git clone https://github.com/TalentCLEF/talentclef25_evaluation_script.git
!pip install -r /content/talentclef25_evaluation_script/requirements.txt


Cloning into 'talentclef25_evaluation_script'...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 27 (delta 10), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 10.10 KiB | 5.05 MiB/s, done.
Resolving deltas: 100% (10/10), done.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/content/talentclef25_evaluation_script/requirements.txt'


Then, select the Qrels file and the Run file to perform the evaluation.


In [23]:
qrels_file = os.path.join(taskB_path, "validation", "qrels.tsv")
run_file = os.path.join(taskB_results_path, "evaluation_baseline_taskB.trec")

In [24]:
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
!python talentclef25_evaluation_script/talentclef_evaluate.py --qrels ~/Documents/projects/data/ir_data/talent_clef/TaskB/validation/qrels.tsv --run ~/Documents/projects/data/ir_data/talent_clef/TaskB/results/evaluation_baseline_taskB.tre

In [27]:
#model.save(taskB_models_path)